In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz_sh = pytz.timezone('Asia/Shanghai')

# helper to read and enrich a dataframe with datetime in UTC+8
def load_df(filename, ts_col='timestamp'):
    df = pd.read_csv(filename)
    # ensure timestamp is numeric
    df[ts_col] = pd.to_numeric(df[ts_col], errors='coerce').astype('Int64')
    # convert epoch seconds (UTC) to tz-aware datetime in Asia/Shanghai
    df['ts_dt'] = pd.to_datetime(df[ts_col], unit='s', utc=True).dt.tz_convert(tz_sh)
    return df

# Load files
metric_df = load_df('metric.csv')
trace_df = load_df('trace.csv')
log_df = load_df('log.csv')
error_df = load_df('error_logs.csv')

# Summaries: total rows, min/max timestamp (as ISO strings in UTC+8)
def summary_df(df):
    total = len(df)
    if total == 0:
        return pd.DataFrame([{'total_rows': 0, 'min_ts': None, 'max_ts': None}])
    min_ts = df['ts_dt'].min()
    max_ts = df['ts_dt'].max()
    return pd.DataFrame([{
        'total_rows': total,
        'min_ts': pd.Timestamp(min_ts).isoformat(),
        'max_ts': pd.Timestamp(max_ts).isoformat()
    }])

metric_summary = summary_df(metric_df)
trace_summary  = summary_df(trace_df)
log_summary    = summary_df(log_df)
error_summary  = summary_df(error_df)

# Top up to 20 cmdb_id by count for each file
metric_top_cmdb = metric_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
trace_top_cmdb  = trace_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
log_top_cmdb    = log_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
error_top_cmdb  = error_df['cmdb_id'].value_counts().head(10).rename_axis('cmdb_id').reset_index(name='count')  # only top 10 for errors

# Unique KPI/trace/log names and their counts (top 20)
metric_kpi_counts_top20  = metric_df['kpi_name'].value_counts().head(20).rename_axis('kpi_name').reset_index(name='count')
trace_name_counts_top20  = trace_df['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')
log_name_counts_top20    = log_df['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')

# For metric: for each of the top 10 kpi_name (by count) show top 10 cmdb_id by count
top10_kpis = metric_kpi_counts_top20.head(10)['kpi_name'].tolist()
rows = []
for k in top10_kpis:
    sub = metric_df[metric_df['kpi_name'] == k]
    vc = sub['cmdb_id'].value_counts().head(10)
    for rank, (cmdb, cnt) in enumerate(vc.items(), start=1):
        rows.append({'kpi_name': k, 'cmdb_id': cmdb, 'count': int(cnt), 'rank': rank})
metric_topkpi_cmdb_top10 = pd.DataFrame(rows).sort_values(['kpi_name','rank']).reset_index(drop=True)

# For trace and log: compact top name lists already prepared (top20)
# For error_logs: up to 10 sample distinct messages with their counts
if 'message' in error_df.columns:
    error_message_counts = error_df['message'].value_counts().head(10).rename_axis('message').reset_index(name='count')
else:
    error_message_counts = pd.DataFrame(columns=['message','count'])

# Compact outputs (ensure not too large). We'll show only needed columns and head limits.
metric_summary, metric_top_cmdb, metric_kpi_counts_top20, metric_topkpi_cmdb_top10.head(200), \
trace_summary, trace_top_cmdb, trace_name_counts_top20, \
log_summary, log_top_cmdb, log_name_counts_top20, \
error_summary, error_top_cmdb, error_message_counts

```
Out[1]:
```
metric_summary, metric_top_cmdb, metric_kpi_counts_top20, metric_topkpi_cmdb_top10, \
trace_summary, trace_top_cmdb, trace_name_counts_top20, \
log_summary, log_top_cmdb, log_name_counts_top20, \
error_summary, error_top_cmdb, error_message_counts

# Build a compact plain-English summary string from the above results
m_total = int(metric_summary.at[0,'total_rows'])
m_min = metric_summary.at[0,'min_ts']
m_max = metric_summary.at[0,'max_ts']
metric_top10_cmdb = metric_top_cmdb.head(10).to_string(index=False)

# pick top 5 KPI names for concise summary and show top 3 cmdb per KPI
top_kpis = metric_kpi_counts_top20.head(10)['kpi_name'].tolist()
kpi_snippets = []
for k in top_kpis:
    sub = metric_topkpi_cmdb_top10[metric_topkpi_cmdb_top10['kpi_name']==k].head(3)
    entries = "; ".join(f"{r['cmdb_id']}({r['count']})" for _, r in sub.iterrows())
    kpi_snippets.append(f"- {k}: {entries}")

t_total = int(trace_summary.at[0,'total_rows'])
t_min = trace_summary.at[0,'min_ts']
t_max = trace_summary.at[0,'max_ts']
trace_top5_cmdb = trace_top_cmdb.head(5).to_string(index=False)
trace_top_names = trace_name_counts_top20.head(8).to_string(index=False)

l_total = int(log_summary.at[0,'total_rows'])
l_min = log_summary.at[0,'min_ts']
l_max = log_summary.at[0,'max_ts']
log_top_cmdb_snip = log_top_cmdb.head(8).to_string(index=False)
log_top_names = log_name_counts_top20.to_string(index=False)

e_total = int(error_summary.at[0,'total_rows'])
e_min = error_summary.at[0,'min_ts']
e_max = error_summary.at[0,'max_ts']
error_top_cmdb_snip = error_top_cmdb.to_string(index=False)
error_messages_snip = error_message_counts.to_string(index=False)

summary = (
    f"Summary of telemetry files (times shown in UTC+8):\n\n"
    f"metric.csv: {m_total} rows; time range {m_min} — {m_max}.\n"
    f"Top cmdb_id (top 10):\n{metric_top10_cmdb}\n\n"
    f"Top KPI names (sample top 10 shown):\n{metric_kpi_counts_top20.head(10).to_string(index=False)}\n\n"
    f"For the top 10 KPI names, example top CMDBs (top 3 per KPI shown):\n" + "\n".join(kpi_snippets) + "\n\n"
    f"trace.csv: {t_total} rows; time range {t_min} — {t_max}.\n"
    f"Top cmdb_id (top 5):\n{trace_top5_cmdb}\n\n"
    f"Top trace names (sample top 8):\n{trace_top_names}\n\n"
    f"log.csv: {l_total} rows; time range {l_min} — {l_max}.\n"
    f"Top cmdb_id (sample top 8):\n{log_top_cmdb_snip}\n"
    f"Top log names:\n{log_top_names}\n\n"
    f"error_logs.csv: {e_total} rows; time range {e_min} — {e_max}.\n"
    f"Top cmdb_id (top 10):\n{error_top_cmdb_snip}\n\n"
    f"Sample distinct error messages (up to 10):\n{error_messages_snip}\n\n"
    f"Notes / highlights:\n"
    f"- metric data is large (147,824 rows) and top KPIs are dominated by container.node-5.* network/memory/cpu metrics.\n"
    f"- trace metrics show many entries coming from frontend pods (e.g., trace.from_frontend-0.* series have ~553 rows each).\n"
    f"- log data mainly records log.error_count and log.row_count.\n"
    f"- error logs are concentrated on frontend: frontend-0 (174), frontend-2 (120), frontend-1 (100); the most frequent message is a warning 'failed to retrieve...' (384 occurrences).\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows                     min_ts                     max_ts
0      147824  2022-03-21T08:00:00+08:00  2022-03-21T08:29:00+08:00,                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                frontend2-0   5280
3                 frontend-1   5040
4                 frontend-0   4920
5                 frontend-2   4620
6          checkoutservice-0   3900
7          checkoutservice-2   3660
8          checkoutservice-1   3540
9         checkoutservice2-0   3540
10   productcatalogservice-2   3480
11   productcatalogservice-0   3480
12   productcatalogservice-1   3408
13              adservice2-0   3060
14             cartservice-1   3000
15  productcatalogservice2-0   2880
16             cartservice-0   2880
17            cartservice2-0   2880
18             cartservice-2   2640
19               adservice-0   2580,                                              kpi_name  count
0   container.node-5.container_network_receive_pac...   1050
1   container.node-5.container_network_transmit_MB...   1050
2   container.node-5.container_network_transmit_er...   1050
3   container.node-5.container_network_transmit_pa...   1050
4   container.node-5.container_network_receive_err...   1050
5   container.node-5.container_network_receive_MB....   1050
6   container.node-5.container_network_receive_pac...   1050
7   container.node-5.container_network_transmit_pa...   1050
8   container.node-5.container_spec_memory_swap_li...    990
9   container.node-5.container_spec_memory_reserva...    990
10    container.node-5.container_tasks_state.sleeping    990
11          container.node-5.container_memory_failcnt    990
12                 container.node-5.container_threads    990
13  container.node-5.container_tasks_state.uninter...    990
14            container.node-5.container_memory_cache    990
15             container.node-5.container_memory_swap    990
16      container.node-5.container_cpu_system_seconds    990
17       container.node-5.container_cpu_usage_seconds    990
18    container.node-5.container_cpu_load_average_10s    990
19        container.node-5.container_cpu_user_seconds    990,                                              kpi_name             cmdb_id  count  rank
0   container.node-5.container_network_receive_MB....         adservice-0     30     1
1   container.node-5.container_network_receive_MB....         adservice-1     30     2
2   container.node-5.container_network_receive_MB....         adservice-2     30     3
3   container.node-5.container_network_receive_MB....       cartservice-0     30     4
4   container.node-5.container_network_receive_MB....       cartservice-1     30     5
..                                                ...                 ...    ...   ...
95  container.node-5.container_spec_memory_swap_li...       cartservice-2     30     6
96  container.node-5.container_spec_memory_swap_li...   checkoutservice-0     30     7
97  container.node-5.container_spec_memory_swap_li...   checkoutservice-1     30     8
98  container.node-5.container_spec_memory_swap_li...   checkoutservice-2     30     9
99  container.node-5.container_spec_memory_swap_li...  checkoutservice2-0     30    10

[100 rows x 4 columns],    total_rows                     min_ts                     max_ts
0       26128  2022-03-21T08:00:00+08:00  2022-03-21T08:29:00+08:00,                      cmdb_id  count
0                 frontend-0   2452
1                 frontend-2   2288
2                 frontend-1   2228
3          checkoutservice-0   1288
4          checkoutservice-2   1172
5                frontend2-0   1044
6          checkoutservice-1   1012
7         checkoutservice2-0    968
8    productcatalogservice-0    968
9    productcatalogservice-1    964
10   productcatalogservice-2    960
11   recommendationservice-2    804
12   recommendationservice-1    796
13   recommendationservice-0    784
14             cartservice-2    628
15             cartservice-1    624
16             cartservice-0    620
17  recommendationservice2-0    580
18         currencyservice-2    556
19         currencyservice-1    556,                                     trace_name  count
0          trace.from_frontend-0.duration_mean    553
1           trace.from_frontend-0.duration_p95    553
2             trace.from_frontend-0.error_rate    553
3              trace.from_frontend-0.row_count    553
4              trace.from_frontend-2.row_count    514
5             trace.from_frontend-2.error_rate    514
6           trace.from_frontend-2.duration_p95    514
7          trace.from_frontend-2.duration_mean    514
8              trace.from_frontend-1.row_count    499
9             trace.from_frontend-1.error_rate    499
10          trace.from_frontend-1.duration_p95    499
11         trace.from_frontend-1.duration_mean    499
12                        trace.self.row_count    430
13                       trace.self.error_rate    430
14                     trace.self.duration_p95    430
15                    trace.self.duration_mean    430
16      trace.from_checkoutservice-0.row_count    253
17     trace.from_checkoutservice-0.error_rate    253
18   trace.from_checkoutservice-0.duration_p95    253
19  trace.from_checkoutservice-0.duration_mean    253,    total_rows                     min_ts                     max_ts
0        1592  2022-03-21T08:00:00+08:00  2022-03-21T08:29:00+08:00,                     cmdb_id  count
0               adservice-0     60
1         currencyservice-2     60
2         currencyservice-0     60
3                frontend-1     60
4                frontend-0     60
5   productcatalogservice-0     60
6                frontend-2     60
7   productcatalogservice-1     60
8   productcatalogservice-2     60
9   recommendationservice-1     60
10  recommendationservice-0     58
11        currencyservice-1     58
12            cartservice-2     58
13            cartservice-1     58
14            cartservice-0     58
15        shippingservice-2     58
16  recommendationservice-2     58
17        shippingservice-0     56
18        shippingservice-1     56
19              adservice-2     56,           log_name  count
0  log.error_count    796
1    log.row_count    796,    total_rows                     min_ts                     max_ts
0         400  2022-03-21T08:00:04+08:00  2022-03-21T08:29:58+08:00,                    cmdb_id  count
0               frontend-0    174
1               frontend-2    120
2               frontend-1    100
3  productcatalogservice-2      3
4  productcatalogservice-0      2
5  productcatalogservice-1      1,                                              message  count
0  severity: warning, message: failed to retrieve...    384
1            severity: error, message: request error      7
2  severity: warning, message: failed to query pr...      2
3  severity: warning, message: failed to query pr...      1
4  "GET /product/1YMWWN1N4O HTTP/1.1" 500 - via_u...      1
5  severity: warning, message: failed to query pr...      1
6  severity: warning, message: failed to query pr...      1
7  severity: warning, message: failed to query pr...      1
8  "GET /product/9SIQT8TOJO HTTP/1.1" 500 - via_u...      1
9  "GET /product/LS4PSXUNUM HTTP/1.1" 500 - via_u...      1)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Use existing metric_df and tz_sh from previous state
import pandas as pd

# 1) Compute global P95 and P99 per kpi_name using the full series (global thresholds)
p95 = metric_df.groupby('kpi_name')['value'].quantile(0.95)
p99 = metric_df.groupby('kpi_name')['value'].quantile(0.99)
thresholds_df = pd.DataFrame({'kpi_name': p95.index, 'p95': p95.values, 'p99': p99.values})

# 2) Filter metric rows to the incident time window in UTC+8
start = pd.Timestamp('2022-03-21 08:00:00', tz=tz_sh)
end   = pd.Timestamp('2022-03-21 08:30:00', tz=tz_sh)
metric_window = metric_df[(metric_df['ts_dt'] >= start) & (metric_df['ts_dt'] <= end)].copy()

# Merge thresholds into the windowed data (so thresholds were computed on full series per rule)
metric_window = metric_window.merge(thresholds_df, on='kpi_name', how='left')

# Compute exceedance flags in-window
metric_window['exceed_p95'] = metric_window['value'] > metric_window['p95']
metric_window['exceed_p99'] = metric_window['value'] > metric_window['p99']

# Aggregate required metrics per kpi_name
agg_counts = metric_window.groupby('kpi_name').agg(
    total_points_in_window = ('value', 'size'),
    count_exceed_p95 = ('exceed_p95', 'sum'),
    count_exceed_p99 = ('exceed_p99', 'sum')
).reset_index()

# Earliest timestamp in-window where value > global P95
earliest = (metric_window[metric_window['exceed_p95']]
            .groupby('kpi_name', sort=True)['ts_dt']
            .min()
            .reset_index()
            .rename(columns={'ts_dt':'earliest_exceed_ts'}))

# Combine aggregates with thresholds and earliest exceed timestamps
result = (agg_counts
          .merge(thresholds_df, on='kpi_name', how='left')
          .merge(earliest, on='kpi_name', how='left'))

# Format earliest_exceed_ts as ISO strings (UTC+8) and keep nulls as None
result['earliest_exceed_ts'] = result['earliest_exceed_ts'].dt.tz_convert(tz_sh)
result['earliest_exceed_ts'] = result['earliest_exceed_ts'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Add node5_flag
result['node5_flag'] = result['kpi_name'].str.contains('node-5', na=False)

# Ensure numeric columns are int where appropriate and round p95/p99 for compactness
result['count_exceed_p95'] = result['count_exceed_p95'].astype(int)
result['count_exceed_p99'] = result['count_exceed_p99'].astype(int)
result['total_points_in_window'] = result['total_points_in_window'].astype(int)
result['p95'] = result['p95'].round(6)
result['p99'] = result['p99'].round(6)

# 3) Return top 20 kpi_name rows sorted by count_exceed_p95 desc, with requested columns
out_cols = ['kpi_name','p95','p99','total_points_in_window','count_exceed_p95',
            'earliest_exceed_ts','count_exceed_p99','node5_flag']
result_df = result.sort_values(['count_exceed_p95','kpi_name'], ascending=[False, True])[out_cols].head(20)

# Display compact table (max 20 rows)
result_df

```
Out[2]:
```
Summary (plain English):

- All top anomalous KPIs are from node-5 (node5_flag = True). The top 20 KPI rows all reference "container.node-5.*", indicating the incident is concentrated on node-5.
- The highest number of P95 exceedances (count_exceed_p95 = 53) are in network-related KPIs (receive/transmit MB and packets). These first exceed P95 at about 08:00–08:02 and have ~11 points above the global P99.
- A second group (count_exceed_p95 ≈ 50) includes CPU-related and memory-failure metrics (cpu cfs periods, cfs throttled seconds/percent, cpu usage seconds, fs usage, memory failure counters). Their earliest exceedances appear between 08:00 and 08:14 and typically ~10 points exceed the global P99.
- Other notable KPIs:
  - container_cpu_system_seconds: 48 exceedances, earliest 08:00.
  - container_sockets: 42 exceedances, earliest 08:00 (no P99 exceedances).
  - memory_cache, memory_rss, memory_usage_MB, memory_working_set_MB: 36–39 exceedances, earliest 08:00–08:11, with some P99 exceedances (~9).
  - container_last_seen peaked near the end of the window (earliest exceed at 08:29) with 33 exceedances.
  - memory failcnt and similar counters have ~30 exceedances starting at 08:00.
- Time window and scale: analysis used the full global series to compute P95/P99, then inspected data in the incident window 2022-03-21 08:00–08:30 (UTC+8). Per-KPI total points in-window are either 1050 or 990 (depending on series), with exceedance counts as reported above.
- Bottom line: node-5 shows simultaneous, broad anomalies across network, CPU, and memory metrics starting around 08:00, with multiple values exceeding the global P95 and a smaller but significant set exceeding P99.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name           p95           p99  total_points_in_window  count_exceed_p95         earliest_exceed_ts  count_exceed_p99  node5_flag
48  container.node-5.container_network_receive_MB....  3.500710e-01  5.744080e-01                    1050                53  2022-03-21T08:00:00+08:00                11        True
50  container.node-5.container_network_receive_pac...  1.391975e+03  2.433815e+03                    1050                53  2022-03-21T08:02:00+08:00                11        True
52  container.node-5.container_network_transmit_MB...  2.993490e+00  3.527559e+00                    1050                53  2022-03-21T08:00:00+08:00                11        True
54  container.node-5.container_network_transmit_pa...  1.309325e+03  2.382070e+03                    1050                53  2022-03-21T08:02:00+08:00                11        True
8          container.node-5.container_cpu_cfs_periods  1.400500e+02  1.921100e+02                     990                50  2022-03-21T08:04:00+08:00                10        True
10  container.node-5.container_cpu_cfs_throttled_s...  6.100000e-03  1.200250e-01                     990                50  2022-03-21T08:00:00+08:00                10        True
13       container.node-5.container_cpu_usage_seconds  2.480920e-01  3.539560e-01                     990                50  2022-03-21T08:21:00+08:00                10        True
14        container.node-5.container_cpu_user_seconds  1.777500e-01  2.155500e-01                     990                50  2022-03-21T08:02:00+08:00                10        True
29   container.node-5.container_fs_usage_MB./dev/vda1  1.141749e+02  1.145794e+02                     990                50  2022-03-21T08:11:00+08:00                10        True
38  container.node-5.container_memory_failures.con...  9.667750e+02  1.487330e+03                     990                50  2022-03-21T08:14:00+08:00                10        True
40  container.node-5.container_memory_failures.hie...  9.667750e+02  1.487330e+03                     990                50  2022-03-21T08:14:00+08:00                10        True
12      container.node-5.container_cpu_system_seconds  8.000000e-02  1.455500e-01                     990                48  2022-03-21T08:00:00+08:00                10        True
56                 container.node-5.container_sockets  1.000000e+01  1.200000e+01                     990                42  2022-03-21T08:00:00+08:00                 0        True
36            container.node-5.container_memory_cache  4.525261e+07  5.566874e+07                     990                39  2022-03-21T08:00:00+08:00                 9        True
44              container.node-5.container_memory_rss  1.368924e+08  1.469972e+08                     990                36  2022-03-21T08:00:00+08:00                 9        True
46         container.node-5.container_memory_usage_MB  1.762617e+02  2.032773e+02                     990                36  2022-03-21T08:00:00+08:00                 9        True
47   container.node-5.container_memory_working_set_MB  1.631016e+02  1.840312e+02                     990                36  2022-03-21T08:00:00+08:00                 9        True
9   container.node-5.container_cpu_cfs_throttled_p...  5.000000e-01  1.555000e+00                     990                34  2022-03-21T08:02:00+08:00                10        True
35               container.node-5.container_last_seen  1.647823e+09  1.647823e+09                     990                33  2022-03-21T08:29:00+08:00                 0        True
37          container.node-5.container_memory_failcnt  9.208463e+06  9.628125e+06                     990                30  2022-03-21T08:00:00+08:00                 0        True```
```

In [ ]:
# In[3]:

# Reuse existing trace_df and tz_sh from prior state
import pandas as pd

# 1) Compute global P95 and P99 per trace_name using the full series
trace_p95 = trace_df.groupby('trace_name')['value'].quantile(0.95)
trace_p99 = trace_df.groupby('trace_name')['value'].quantile(0.99)
trace_thresholds = pd.DataFrame({
    'trace_name': trace_p95.index,
    'p95': trace_p95.values,
    'p99': trace_p99.values
}).reset_index(drop=True)

# 2) Filter trace rows to the incident time window (UTC+8)
start = pd.Timestamp('2022-03-21 08:00:00', tz=tz_sh)
end   = pd.Timestamp('2022-03-21 08:30:00', tz=tz_sh)
trace_window = trace_df[(trace_df['ts_dt'] >= start) & (trace_df['ts_dt'] <= end)].copy()

# Merge thresholds into windowed data
trace_window = trace_window.merge(trace_thresholds, on='trace_name', how='left')

# Exceedance flags based on global thresholds computed above
trace_window['exceed_p95'] = trace_window['value'] > trace_window['p95']
trace_window['exceed_p99'] = trace_window['value'] > trace_window['p99']

# Aggregate per trace_name within the window
agg = trace_window.groupby('trace_name').agg(
    total_points_in_window = ('value','size'),
    count_exceed_p95 = ('exceed_p95','sum'),
    count_exceed_p99 = ('exceed_p99','sum')
).reset_index()

# Earliest timestamp in-window where value > global P95
earliest = (trace_window[trace_window['exceed_p95']]
            .groupby('trace_name')['ts_dt']
            .min()
            .reset_index()
            .rename(columns={'ts_dt':'earliest_exceed_ts'}))

# Combine aggregates with thresholds and earliest exceed timestamps
trace_result = (agg
                .merge(trace_thresholds, on='trace_name', how='left')
                .merge(earliest, on='trace_name', how='left'))

# Format earliest_exceed_ts as ISO strings in UTC+8, null otherwise
trace_result['earliest_exceed_ts'] = trace_result['earliest_exceed_ts'].dt.tz_convert(tz_sh)
trace_result['earliest_exceed_ts'] = trace_result['earliest_exceed_ts'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Round thresholds and ensure integer counts
trace_result['p95'] = trace_result['p95'].round(6)
trace_result['p99'] = trace_result['p99'].round(6)
trace_result['total_points_in_window'] = trace_result['total_points_in_window'].astype(int)
trace_result['count_exceed_p95'] = trace_result['count_exceed_p95'].astype(int)
trace_result['count_exceed_p99'] = trace_result['count_exceed_p99'].astype(int)

# 3) Top 20 trace_name rows sorted by count_exceed_p95 desc
trace_table = trace_result.sort_values(['count_exceed_p95','trace_name'], ascending=[False, True]) \
                         [['trace_name','p95','p99','total_points_in_window','count_exceed_p95','earliest_exceed_ts','count_exceed_p99']] \
                         .head(20)

# 4) For trace_names with count_exceed_p95>0, pick top 3 by count_exceed_p95 and get up to earliest 5 exceed rows each
top3_names = trace_table[trace_table['count_exceed_p95']>0].nlargest(3, 'count_exceed_p95')['trace_name'].tolist()

# Collect up to 5 earliest exceed rows per selected trace_name
examples_list = []
for name in top3_names:
    sub = trace_window[(trace_window['trace_name']==name) & (trace_window['exceed_p95'])].copy()
    sub = sub.sort_values('ts_dt').head(5)
    if not sub.empty:
        # prepare requested columns: timestamp (ISO), cmdb_id, trace_name, value
        sub_out = sub.loc[:, ['ts_dt','cmdb_id','trace_name','value']].copy()
        sub_out['timestamp'] = sub_out['ts_dt'].dt.tz_convert(tz_sh).apply(lambda x: x.isoformat())
        sub_out = sub_out[['timestamp','cmdb_id','trace_name','value']]
        examples_list.append(sub_out)

if examples_list:
    trace_examples = pd.concat(examples_list, ignore_index=True).head(15)  # ensure overall cap 15
else:
    trace_examples = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])

# Final compact outputs: trace_table (max 20 rows) and trace_examples (up to 15 rows)
trace_table, trace_examples

```
Out[3]:
```
Summary (plain English):

- Analysis window: 2022-03-21 08:00:00 — 2022-03-21 08:30:00 (UTC+8). Global P95/P99 were computed on each trace_name using the full series before filtering.

- Top trace anomalies (sorted by number of points in-window exceeding global P95):
  1. trace.from_frontend-0.duration_mean — p95=0.038170, p99=0.059699, 553 points in window, 28 > P95, earliest exceed 2022-03-21T08:00:00+08:00, 6 > P99.
  2. trace.from_frontend-0.duration_p95 — p95=0.038114, p99=0.048494, 553 pts, 28 > P95, earliest 2022-03-21T08:00:00+08:00, 6 > P99.
  3. trace.from_frontend-0.row_count — p95=47.0, p99=65.96, 553 pts, 27 > P95, earliest 2022-03-21T08:05:00+08:00, 6 > P99.
  4. trace.from_frontend-2.duration_mean — p95=0.004558, p99=0.040609, 514 pts, 26 > P95, earliest 08:01, 6 > P99.
  5. trace.from_frontend-2.duration_p95 — p95=0.005980, p99=0.040345, 514 pts, 26 > P95, earliest 08:01, 6 > P99.
  6. trace.from_frontend-1.duration_mean — p95=0.004487, p99=0.076551, 499 pts, 25 > P95, earliest 08:00, 5 > P99.
  7. trace.from_frontend-1.duration_p95 — p95=0.005807, p99=0.076551, 499 pts, 25 > P95, earliest 08:00, 5 > P99.
  8. trace.from_frontend-1.row_count — p95=35.0, p99=48.02, 499 pts, 24 > P95, earliest 08:21, 5 > P99.
  9. trace.from_frontend-2.row_count — p95=44.0, p99=66.87, 514 pts, 24 > P95, earliest 08:21, 6 > P99.
  10. trace.self.duration_mean — p95=0.00718, p99≈23.9744, 430 pts, 22 > P95, earliest 08:10, 5 > P99.
  (table continues up to 20 rows; all above are the most affected.)

- Pattern: the most impacted trace_names are "trace.from_frontend-<pod>.*" series (frontend→backend traces). These have the highest counts of values exceeding global P95 (many starts at ~08:00—08:02) and several points also exceed global P99.

- Example exceed events (earliest examples for the top 3 trace_names; up to 5 earliest rows each):
  - trace.from_frontend-0.duration_mean: exceed rows at 08:00 from checkoutservice-0 and checkoutservice-2, then 08:02 from checkoutservice-0 and checkoutservice-1, etc.
  - trace.from_frontend-0.duration_p95: same early exceed rows at 08:00 and 08:02 on checkoutservice pods.
  - trace.from_frontend-0.row_count: exceed row_count events at 08:05 on productcatalogservice-0/1/2 and later at 08:21 (productcatalogservice-0/1).

- Conclusion: The trace anomalies are concentrated on frontend-originated traces, with the strongest signals toward checkoutservice and productcatalogservice between 08:00–08:21. Multiple trace metrics exceed their global P95 (and some exceed P99), indicating elevated latency/row_count activity during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(                                         trace_name         p95         p99  total_points_in_window  count_exceed_p95         earliest_exceed_ts  count_exceed_p99
16              trace.from_frontend-0.duration_mean    0.038170    0.059699                     553                28  2022-03-21T08:00:00+08:00                 6
17               trace.from_frontend-0.duration_p95    0.038114    0.048494                     553                28  2022-03-21T08:00:00+08:00                 6
19                  trace.from_frontend-0.row_count   47.000000   65.960000                     553                27  2022-03-21T08:05:00+08:00                 6
24              trace.from_frontend-2.duration_mean    0.004558    0.040609                     514                26  2022-03-21T08:01:00+08:00                 6
25               trace.from_frontend-2.duration_p95    0.005980    0.040345                     514                26  2022-03-21T08:01:00+08:00                 6
20              trace.from_frontend-1.duration_mean    0.004487    0.076551                     499                25  2022-03-21T08:00:00+08:00                 5
21               trace.from_frontend-1.duration_p95    0.005807    0.076551                     499                25  2022-03-21T08:00:00+08:00                 5
23                  trace.from_frontend-1.row_count   35.000000   48.020000                     499                24  2022-03-21T08:21:00+08:00                 5
27                  trace.from_frontend-2.row_count   44.000000   66.870000                     514                24  2022-03-21T08:21:00+08:00                 6
52                         trace.self.duration_mean    0.007180   23.974403                     430                22  2022-03-21T08:10:00+08:00                 5
53                          trace.self.duration_p95    0.039367   41.056608                     430                22  2022-03-21T08:00:00+08:00                 5
55                             trace.self.row_count  404.100000  786.490000                     430                22  2022-03-21T08:00:00+08:00                 5
0        trace.from_checkoutservice-0.duration_mean    0.004709    0.005465                     253                13  2022-03-21T08:00:00+08:00                 3
1         trace.from_checkoutservice-0.duration_p95    0.004719    0.005465                     253                13  2022-03-21T08:00:00+08:00                 3
152  trace.to_productcatalogservice-0.duration_mean    0.004763    0.742622                     242                13  2022-03-21T08:00:00+08:00                 3
153   trace.to_productcatalogservice-0.duration_p95    0.005703    2.514065                     242                13  2022-03-21T08:02:00+08:00                 3
8        trace.from_checkoutservice-2.duration_mean    0.004753    0.005460                     229                12  2022-03-21T08:02:00+08:00                 3
9         trace.from_checkoutservice-2.duration_p95    0.004762    0.005469                     229                12  2022-03-21T08:02:00+08:00                 3
156  trace.to_productcatalogservice-1.duration_mean    0.004722    2.026283                     241                12  2022-03-21T08:00:00+08:00                 3
157   trace.to_productcatalogservice-1.duration_p95    0.005625    0.006033                     241                12  2022-03-21T08:01:00+08:00                 3,                     timestamp                  cmdb_id                           trace_name      value
0   2022-03-21T08:00:00+08:00        checkoutservice-0  trace.from_frontend-0.duration_mean   0.038601
1   2022-03-21T08:00:00+08:00        checkoutservice-2  trace.from_frontend-0.duration_mean   0.039686
2   2022-03-21T08:02:00+08:00        checkoutservice-0  trace.from_frontend-0.duration_mean   0.053231
3   2022-03-21T08:02:00+08:00        checkoutservice-1  trace.from_frontend-0.duration_mean   0.041083
4   2022-03-21T08:04:00+08:00        checkoutservice-2  trace.from_frontend-0.duration_mean   0.038680
5   2022-03-21T08:00:00+08:00        checkoutservice-0   trace.from_frontend-0.duration_p95   0.038601
6   2022-03-21T08:00:00+08:00        checkoutservice-2   trace.from_frontend-0.duration_p95   0.039686
7   2022-03-21T08:02:00+08:00        checkoutservice-0   trace.from_frontend-0.duration_p95   0.053231
8   2022-03-21T08:02:00+08:00        checkoutservice-1   trace.from_frontend-0.duration_p95   0.041083
9   2022-03-21T08:04:00+08:00        checkoutservice-2   trace.from_frontend-0.duration_p95   0.038680
10  2022-03-21T08:05:00+08:00  productcatalogservice-0      trace.from_frontend-0.row_count  51.000000
11  2022-03-21T08:05:00+08:00  productcatalogservice-1      trace.from_frontend-0.row_count  52.000000
12  2022-03-21T08:05:00+08:00  productcatalogservice-2      trace.from_frontend-0.row_count  52.000000
13  2022-03-21T08:21:00+08:00  productcatalogservice-0      trace.from_frontend-0.row_count  64.000000
14  2022-03-21T08:21:00+08:00  productcatalogservice-1      trace.from_frontend-0.row_count  63.000000)```
```

In [ ]:
# In[4]:

# Reuse metric_df and tz_sh from previous state
import pandas as pd

# 1) Select kpi_names containing 'container.node-5' and compute global P95 (on full series)
node5_mask = metric_df['kpi_name'].str.contains('container.node-5', na=False)
node5_metrics = metric_df[node5_mask]

# Compute global P95 per kpi_name (must be computed on full series before window filtering)
p95_series = node5_metrics.groupby('kpi_name')['value'].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={'value':'p95'})

# 2) Filter metric rows to the incident window (UTC+8)
start = pd.Timestamp('2022-03-21 08:00:00', tz=tz_sh)
end   = pd.Timestamp('2022-03-21 08:30:00', tz=tz_sh)
window_mask = (metric_df['ts_dt'] >= start) & (metric_df['ts_dt'] <= end)
window_node5 = metric_df[window_mask & node5_mask].copy()

# Merge global P95 into the windowed node-5 rows
window_node5 = window_node5.merge(p95_df, on='kpi_name', how='left')

# Mark exceedances where value > global P95
window_node5['is_exceed'] = window_node5['value'] > window_node5['p95']

# Keep only exceedance rows
exceed_df = window_node5[window_node5['is_exceed']].copy()

# 3) Group exceedances by cmdb_id and compute required aggregates
if not exceed_df.empty:
    grouped = (exceed_df.groupby('cmdb_id')
               .agg(
                   exceed_count = ('is_exceed','size'),
                   earliest_exceed_ts = ('ts_dt','min'),
                   affected_kpi_count = ('kpi_name','nunique')
               )
               .reset_index())
    # Format earliest_exceed_ts as ISO string in UTC+8
    grouped['earliest_exceed_ts'] = grouped['earliest_exceed_ts'].dt.tz_convert(tz_sh).apply(lambda x: x.isoformat())
    # Sort and limit to top 20
    top_cmdb = grouped.sort_values(['exceed_count','cmdb_id'], ascending=[False, True]).head(20)
    total_distinct_cmdb = int(grouped['cmdb_id'].nunique())
else:
    # empty results
    top_cmdb = pd.DataFrame(columns=['cmdb_id','exceed_count','earliest_exceed_ts','affected_kpi_count'])
    total_distinct_cmdb = 0

# Ensure column order
top_cmdb = top_cmdb[['cmdb_id','exceed_count','affected_kpi_count','earliest_exceed_ts']]

# Return compact table (max 20 rows) and the scalar count of distinct cmdb_id with exceedances
top_cmdb, total_distinct_cmdb

```
Out[4]:
```
Summary (plain English):

- Analysis focused on KPIs containing "container.node-5". Global P95 was computed per KPI on the full series, then exceedances in the incident window 2022-03-21 08:00–08:30 (UTC+8) were counted and grouped by cmdb_id.

- Total distinct cmdb_id that had at least one exceedance: 34.

- Top cmdb_id by exceed_count (top 20 shown; columns: cmdb_id — exceed_count — affected_kpi_count — earliest_exceed_ts):
  1. adservice-0 — 145 — 23 — 2022-03-21T08:00:00+08:00
  2. frontend-0 — 86 — 9 — 2022-03-21T08:00:00+08:00
  3. productcatalogservice-0 — 86 — 9 — 2022-03-21T08:02:00+08:00
  4. productcatalogservice-1 — 85 — 9 — 2022-03-21T08:00:00+08:00
  5. productcatalogservice-2 — 83 — 9 — 2022-03-21T08:00:00+08:00
  6. emailservice-1 — 62 — 4 — 2022-03-21T08:00:00+08:00
  7. adservice-1 — 61 — 22 — 2022-03-21T08:27:00+08:00
  8. cartservice-0 — 61 — 7 — 2022-03-21T08:00:00+08:00
  9. adservice-2 — 60 — 23 — 2022-03-21T08:16:00+08:00
 10. currencyservice-0 — 52 — 7 — 2022-03-21T08:00:00+08:00
 11. frontend-2 — 42 — 8 — 2022-03-21T08:10:00+08:00
 12. paymentservice-0 — 32 — 3 — 2022-03-21T08:00:00+08:00
 13. cartservice-2 — 27 — 7 — 2022-03-21T08:00:00+08:00
 14. frontend-1 — 27 — 6 — 2022-03-21T08:02:00+08:00
 15. checkoutservice2-0 — 26 — 4 — 2022-03-21T08:00:00+08:00
 16. cartservice-1 — 24 — 5 — 2022-03-21T08:02:00+08:00
 17. currencyservice-2 — 22 — 4 — 2022-03-21T08:11:00+08:00
 18. currencyservice-1 — 18 — 4 — 2022-03-21T08:14:00+08:00
 19. checkoutservice-1 — 17 — 3 — 2022-03-21T08:05:00+08:00
 20. checkoutservice-2 — 14 — 4 — 2022-03-21T08:09:00+08:00

- Key observations:
  - Many services/pods experienced exceedances of node-5 related KPIs during the window; adservice-0 is the most affected (145 exceedances across 23 distinct node-5 KPIs).
  - Several cmdb_ids show earliest exceedances at the window start (08:00), indicating the issue began at or before that time for multiple components.
  - A broad set of services (frontend, productcatalogservice, adservice, cartservice, currencyservice, checkoutservice, etc.) show exceedances, meaning node-5-related metric anomalies impacted many pods/services.

- Short conclusion: node-5 related KPI thresholds were frequently exceeded across many pods during 08:00–08:30, with adservice-0 and multiple frontend/productcatalogservice pods among the most affected.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  exceed_count  affected_kpi_count         earliest_exceed_ts
0               adservice-0           145                  23  2022-03-21T08:00:00+08:00
17               frontend-0            86                   9  2022-03-21T08:00:00+08:00
23  productcatalogservice-0            86                   9  2022-03-21T08:02:00+08:00
24  productcatalogservice-1            85                   9  2022-03-21T08:00:00+08:00
25  productcatalogservice-2            83                   9  2022-03-21T08:00:00+08:00
14           emailservice-1            62                   4  2022-03-21T08:00:00+08:00
1               adservice-1            61                  22  2022-03-21T08:27:00+08:00
3             cartservice-0            61                   7  2022-03-21T08:00:00+08:00
2               adservice-2            60                  23  2022-03-21T08:16:00+08:00
10        currencyservice-0            52                   7  2022-03-21T08:00:00+08:00
19               frontend-2            42                   8  2022-03-21T08:10:00+08:00
20         paymentservice-0            32                   3  2022-03-21T08:00:00+08:00
5             cartservice-2            27                   7  2022-03-21T08:00:00+08:00
18               frontend-1            27                   6  2022-03-21T08:02:00+08:00
9        checkoutservice2-0            26                   4  2022-03-21T08:00:00+08:00
4             cartservice-1            24                   5  2022-03-21T08:02:00+08:00
12        currencyservice-2            22                   4  2022-03-21T08:11:00+08:00
11        currencyservice-1            18                   4  2022-03-21T08:14:00+08:00
7         checkoutservice-1            17                   3  2022-03-21T08:05:00+08:00
8         checkoutservice-2            14                   4  2022-03-21T08:09:00+08:00, 34)```
```

In [ ]:
# In[5]:

# Reuse metric_df and tz_sh from previous state
import pandas as pd
import numpy as np

# 1) Select kpi_names containing 'container.node-5' and compute global P95 and P99 on full series
node5_mask = metric_df['kpi_name'].str.contains('container.node-5', na=False)
node5_full = metric_df[node5_mask]

p95_series = node5_full.groupby('kpi_name')['value'].quantile(0.95)
p99_series = node5_full.groupby('kpi_name')['value'].quantile(0.99)
thresholds = pd.DataFrame({
    'kpi_name': p95_series.index,
    'p95': p95_series.values,
    'p99': p99_series.values
})

# 2) Filter metric rows to the incident window (UTC+8)
start = pd.Timestamp('2022-03-21 08:00:00', tz=tz_sh)
end   = pd.Timestamp('2022-03-21 08:30:00', tz=tz_sh)
window_mask = (metric_df['ts_dt'] >= start) & (metric_df['ts_dt'] <= end) & node5_mask
window_node5 = metric_df[window_mask].copy()

# Merge thresholds (global) into windowed rows
window_node5 = window_node5.merge(thresholds, on='kpi_name', how='left')

# Mark exceedances vs global P95/P99
window_node5['exceed_p95'] = window_node5['value'] > window_node5['p95']
window_node5['exceed_p99'] = window_node5['value'] > window_node5['p99']

# 3) Aggregate per kpi_name for required metrics
# total_points_in_window, count_exceed_p95, count_exceed_p99, earliest_exceed_ts, max_value_in_window, top_cmdb_id_with_max_value, max_over_p95_ratio
grouped = window_node5.groupby('kpi_name')

agg_list = []
for name, grp in grouped:
    total_points = int(grp.shape[0])
    cnt_p95 = int(grp['exceed_p95'].sum())
    cnt_p99 = int(grp['exceed_p99'].sum())
    # earliest exceed ts for this kpi_name (value > p95)
    if cnt_p95 > 0:
        earliest_ts = grp.loc[grp['exceed_p95'], 'ts_dt'].min().tz_convert(tz_sh).isoformat()
    else:
        earliest_ts = None
    # max value in window and corresponding cmdb_id (take first if multiple)
    if not grp.empty:
        idxmax = grp['value'].idxmax()
        max_val = float(grp.loc[idxmax, 'value'])
        top_cmdb = grp.loc[idxmax, 'cmdb_id']
    else:
        max_val = np.nan
        top_cmdb = None
    # find p95 (same for all rows of this kpi_name)
    p95_val = float(grp['p95'].iloc[0]) if 'p95' in grp.columns and pd.notnull(grp['p95'].iloc[0]) else np.nan
    # ratio (handle zero/NaN p95)
    if pd.notnull(p95_val) and p95_val != 0:
        ratio = float(max_val / p95_val)
    else:
        ratio = np.nan
    agg_list.append({
        'kpi_name': name,
        'p95': p95_val,
        'p99': float(grp['p99'].iloc[0]) if 'p99' in grp.columns and pd.notnull(grp['p99'].iloc[0]) else np.nan,
        'total_points_in_window': total_points,
        'count_exceed_p95': cnt_p95,
        'count_exceed_p99': cnt_p99,
        'earliest_exceed_ts': earliest_ts,
        'max_value_in_window': max_val,
        'max_over_p95_ratio': ratio,
        'top_cmdb_id_with_max_value': top_cmdb
    })

agg_df = pd.DataFrame(agg_list)

# 4) Top 10 kpi_name rows sorted by max_over_p95_ratio desc
agg_df_sorted = agg_df.sort_values('max_over_p95_ratio', ascending=False).head(10)

# Round numeric columns for compactness
agg_df_sorted['p95'] = agg_df_sorted['p95'].round(6)
agg_df_sorted['p99'] = agg_df_sorted['p99'].round(6)
agg_df_sorted['max_value_in_window'] = agg_df_sorted['max_value_in_window'].round(6)
agg_df_sorted['max_over_p95_ratio'] = agg_df_sorted['max_over_p95_ratio'].round(3)

top10_table = agg_df_sorted[[
    'kpi_name','p95','p99','total_points_in_window','count_exceed_p95',
    'count_exceed_p99','earliest_exceed_ts','max_value_in_window',
    'max_over_p95_ratio','top_cmdb_id_with_max_value'
]]

# 5) Single-row summary: overall earliest_exceed_ts across all node-5 exceedances and corresponding kpi_name and cmdb_id
# Find earliest row in window_node5 where exceed_p95 is True
exceed_rows = window_node5[window_node5['exceed_p95']].copy()
if not exceed_rows.empty:
    earliest_row_idx = exceed_rows['ts_dt'].idxmin()
    earliest_row = exceed_rows.loc[earliest_row_idx]
    overall_earliest_summary = pd.DataFrame([{
        'overall_earliest_exceed_ts': earliest_row['ts_dt'].tz_convert(tz_sh).isoformat(),
        'kpi_name': earliest_row['kpi_name'],
        'cmdb_id': earliest_row['cmdb_id']
    }])
else:
    overall_earliest_summary = pd.DataFrame([{
        'overall_earliest_exceed_ts': None,
        'kpi_name': None,
        'cmdb_id': None
    }])

# Display compact outputs: top10_table (max 10 rows) and overall_earliest_summary (single row)
top10_table, overall_earliest_summary

```
Out[5]:
```
Summary (plain English):

- Earliest exceedance overall: 2022-03-21T08:00:00+08:00 — KPI: container.node-5.container_memory_rss on cmdb_id adservice-0.

- Top 10 node-5 KPIs by how far their in-window max exceeded the global P95 (max_over_p95_ratio):
  1. container_cpu_cfs_throttled_seconds — p95=0.0061, p99=0.120025; 990 pts in window; 50 >P95, 10 >P99; earliest exceed 08:00; max=6.496037 (≈1065× P95); top cmdb: adservice-1.
  2. container_cpu_cfs_throttled_percent — p95=0.5, p99=1.555; 990 pts; 34 >P95, 10 >P99; earliest 08:02; max=131.5 (≈263×); top cmdb: adservice-1.
  3. container_cpu_system_seconds — p95=0.08, p99=0.14555; 990 pts; 48 >P95, 10 >P99; earliest 08:00; max=8.755 (≈109×); top cmdb: adservice-1.
  4. container_fs_usage_MB./dev/vda1 — p95≈114.1749, p99≈114.5794; 990 pts; 50 >P95, 10 >P99; earliest 08:11; max≈6925.676 (≈60.7×); top cmdb: adservice-0.
  5. container_memory_failures.count (and related high-water metrics) — p95≈966.78, p99≈1487.33; 990 pts; 50 >P95, 10 >P99; earliest 08:14; max≈51181.5 (≈52.9×); top cmdb: adservice-1.
  6. container_memory_failures.highest (same pattern) — similar stats and top cmdb adservice-1.
  7. container_cpu_usage_seconds — p95≈0.24809, p99≈0.35396; 990 pts; 50 >P95, 10 >P99; earliest 08:21; max≈10.24488 (≈41.3×); top cmdb: adservice-1.
  8. container_memory_cache — p95≈4.525e7, p99≈5.567e7; 990 pts; 39 >P95, 9 >P99; earliest 08:00; max≈3.00007e8 (≈6.63×); top cmdb: adservice-2.
  9. container_cpu_user_seconds — p95≈0.17775, p99≈0.21555; 990 pts; 50 >P95, 10 >P99; earliest 08:02; max≈0.965 (≈5.43×); top cmdb: adservice-0.
 10. container_cpu_cfs_periods — p95≈140.05, p99≈192.11; 990 pts; 50 >P95, 10 >P99; earliest 08:04; max≈579.5 (≈4.14×); top cmdb: adservice-1.

- Key takeaways:
  - Severe exceedances center on CPU throttling/usage, memory failures and FS usage for metrics tied to node-5.
  - adservice-1 and adservice-0 frequently appear as the cmdb_id where extreme max values occurred.
  - The largest relative spikes vs. global P95 are extreme (hundreds to >1000×) for CPU throttling metrics, indicating brief but very large outliers during the incident window.
  - Many of these exceedances begin at or shortly after 08:00 (UTC+8), with some specific KPIs peaking later (e.g., fs usage ~08:11, memory failure counters ~08:14).

This summary is based on global P95/P99 computed on the full series, then analyzing the 08:00–08:30 window.

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name           p95           p99  total_points_in_window  count_exceed_p95  count_exceed_p99         earliest_exceed_ts  max_value_in_window  max_over_p95_ratio top_cmdb_id_with_max_value
2   container.node-5.container_cpu_cfs_throttled_s...  6.100000e-03  1.200250e-01                     990                50                10  2022-03-21T08:00:00+08:00         6.496037e+00            1064.913                adservice-1
1   container.node-5.container_cpu_cfs_throttled_p...  5.000000e-01  1.555000e+00                     990                34                10  2022-03-21T08:02:00+08:00         1.315000e+02             263.000                adservice-1
4       container.node-5.container_cpu_system_seconds  8.000000e-02  1.455500e-01                     990                48                10  2022-03-21T08:00:00+08:00         8.755000e+00             109.438                adservice-1
21   container.node-5.container_fs_usage_MB./dev/vda1  1.141749e+02  1.145794e+02                     990                50                10  2022-03-21T08:11:00+08:00         6.925676e+03              60.658                adservice-0
30  container.node-5.container_memory_failures.con...  9.667750e+02  1.487330e+03                     990                50                10  2022-03-21T08:14:00+08:00         5.118150e+04              52.940                adservice-1
32  container.node-5.container_memory_failures.hie...  9.667750e+02  1.487330e+03                     990                50                10  2022-03-21T08:14:00+08:00         5.118150e+04              52.940                adservice-1
5        container.node-5.container_cpu_usage_seconds  2.480920e-01  3.539560e-01                     990                50                10  2022-03-21T08:21:00+08:00         1.024488e+01              41.295                adservice-1
28            container.node-5.container_memory_cache  4.525261e+07  5.566874e+07                     990                39                 9  2022-03-21T08:00:00+08:00         3.000074e+08               6.630                adservice-2
6         container.node-5.container_cpu_user_seconds  1.777500e-01  2.155500e-01                     990                50                10  2022-03-21T08:02:00+08:00         9.650000e-01               5.429                adservice-0
0          container.node-5.container_cpu_cfs_periods  1.400500e+02  1.921100e+02                     990                50                10  2022-03-21T08:04:00+08:00         5.795000e+02               4.138                adservice-1,   overall_earliest_exceed_ts                               kpi_name      cmdb_id
0  2022-03-21T08:00:00+08:00  container.node-5.container_memory_rss  adservice-0)```
```